# Autoformer 직접 구현 및 재현 (NeurIPS 2021)

**논문**: Wu et al., *Autoformer: Decomposition Transformers with Auto-Correlation for Long-Term Series Forecasting*

이 노트북은 공식 코드를 import하지 않고 **데이터 로더, 모델, 학습 루프를 모두 직접 구현**합니다.
각 부분이 논문의 어느 식에 대응하는지 설명과 함께 정리했습니다.

| 섹션 | 내용 | 논문 대응 |
|---|---|---|
| 0 | 환경 설정 | |
| 1 | 데이터 로더 (분할, 정규화, 시간 특징, 슬라이딩 윈도우) | 4장 Datasets |
| 2 | Series Decomposition 블록 | 3.1절, 식 (1) |
| 3 | Auto-Correlation 메커니즘 | 3.2절, 식 (5)~(8), Algorithm 3·4 |
| 4 | 임베딩 (위치 임베딩 없음) | Appendix G.2 |
| 5 | 인코더 · 디코더 | 식 (3), (4) |
| 6 | 전체 모델 (디코더 입력 초기화) | 식 (2), Figure 1 |
| 7 | (선택) 공식 코드와 출력 일치 검증 | |
| 8 | 학습 · 평가 루프 | 4장 Implementation details |
| 9 | 실험 실행 | Table 1, Table 10 |
| 10 | 결과 정리 (논문 수치와 비교) | Table 10 |
| 11 | 시각화 (예측, 분해, 학습된 주기) | Figure 4, 6, 8 |

**사용 순서**
1. 런타임을 GPU로 설정합니다 (런타임 → 런타임 유형 변경).
2. 섹션 0의 `QUICK_TEST = True`로 두고 **전체 실행**(런타임 → 모두 실행)해서 오류 없이 끝나는지 먼저 확인합니다. 몇 분이면 끝납니다.
3. 확인되면 `QUICK_TEST = False`로 바꾸고 섹션 9의 실험을 실행합니다.

## 0. 환경 설정

- Google Drive를 연결하고 작업 폴더를 지정합니다. `WORK_DIR`는 본인 폴더 이름으로 바꿔 주세요.
- 데이터는 공식 저장소를 clone할 때 만든 `Autoformer/dataset` 폴더를 그대로 사용합니다.
- 결과(CSV, 체크포인트)는 `WORK_DIR/my_autoformer/`에 저장되므로 세션이 끊겨도 남습니다.

In [ ]:
import os, sys

# ▼ 본인 폴더 이름으로 수정
WORK_DIR = '/content/drive/MyDrive/황대경_Autoformer'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    # Colab이 아닌 환경(로컬 테스트)에서는 환경 변수로 경로 지정
    WORK_DIR = os.environ.get('AUTOFORMER_WORK_DIR', WORK_DIR)

DATA_ROOT = os.path.join(WORK_DIR, 'Autoformer', 'dataset')   # 데이터 위치
OFFICIAL_DIR = os.path.join(WORK_DIR, 'Autoformer')            # 섹션 7 검증용 공식 코드 위치
OUT_DIR = os.path.join(WORK_DIR, 'my_autoformer')              # 내 구현의 결과 저장 위치
FIG_DIR = os.path.join(OUT_DIR, 'figures')                     # 그래프 PNG 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

# True: 아주 짧게 돌려서 전체 셀이 오류 없이 실행되는지만 확인 (결과 수치는 의미 없음)
# False: 논문 설정 그대로 실험
QUICK_TEST = True

print('DATA_ROOT 존재:', os.path.isdir(DATA_ROOT))
print('OUT_DIR:', OUT_DIR)

def save_fig(name):
    # 현재 그래프를 FIG_DIR/name.png로 저장 (발표자료·README용)
    path = os.path.join(FIG_DIR, f'{name}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print('저장:', path)

In [ ]:
import math, time, copy, random, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch', torch.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

def set_seed(seed=2021):
    # 공식 코드 run.py와 같은 시드(2021)를 사용
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. 데이터 로더

### 1-1. 데이터 분할
논문 4장은 ETT를 6:2:2, 나머지를 7:1:2로 시간 순서대로 나눈다고 설명합니다. 실제 구현은 다음과 같습니다.

| 데이터 | 학습 | 검증 | 테스트 |
|---|---|---|---|
| ETT (ETTm1, ETTm2: 15분 간격) | 처음 12개월 | 다음 4개월 | 다음 4개월 |
| ETT (ETTh1, ETTh2: 1시간 간격) | 처음 12개월 | 다음 4개월 | 다음 4개월 |
| 나머지 (custom) | 앞 70% | 중간 10% | 뒤 20% |

한 달을 30일로 계산하므로 ETTm2의 학습 구간은 $12 \times 30 \times 24 \times 4 = 34{,}560$개 시점입니다.
검증·테스트 구간은 시작점을 `seq_len`만큼 앞당겨서, 첫 샘플도 입력 96개를 온전히 가질 수 있게 합니다.

### 1-2. 정규화
**학습 구간의 평균과 표준편차만으로** 전체 데이터를 표준화합니다. 검증·테스트 데이터의 통계를 쓰면 미래 정보가 새어 들어가기 때문입니다.
논문의 MSE·MAE는 모두 **정규화된 값 기준**입니다.

### 1-3. 시간 특징 (time feature)
각 시점의 날짜·시각을 $[-0.5, 0.5]$ 범위의 숫자로 바꿔 모델에 함께 넣습니다. `freq`에 따라 사용하는 특징이 다릅니다.

| freq | 특징 (순서대로) | 개수 |
|---|---|---|
| `'t'` (분 단위, ETTm 공식 설정) | 분, 시, 요일, 일, 연중 일자 | 5 |
| `'h'` (시간 단위, 기본값) | 시, 요일, 일, 연중 일자 | 4 |
| `'d'` (일 단위) | 요일, 일, 연중 일자 | 3 |

공식 코드는 pandas의 `to_offset('t')`로 freq를 해석하는데, 최신 pandas에서는 `'t'` 표기가 제거되어 오류가 날 수 있습니다.
여기서는 같은 값을 직접 계산하므로 pandas 버전과 무관하게 동작합니다.

### 1-4. 슬라이딩 윈도우
샘플 하나는 아래 네 개로 구성됩니다 (`seq_len=96, label_len=48, pred_len=96` 기준).

```
시간축:  |------------- x (96) -------------|
                          |-- y 앞부분 (48) --|----- y 예측 구간 (96) -----|
```
- `x`: 인코더 입력 (과거 96개)
- `y`: 디코더 구간 (입력의 마지막 48개 + 예측할 96개). 손실은 마지막 96개에서만 계산합니다.
- `x_mark`, `y_mark`: 각 구간의 시간 특징

샘플 수 = 구간 길이 − `seq_len` − `pred_len` + 1 입니다.

In [ ]:
def time_features(dates: pd.DatetimeIndex, freq: str) -> np.ndarray:
    # 공식 코드 utils/timefeatures.py와 같은 값을 직접 계산 → [T, 특징 수]
    feats = {
        'minute':     dates.minute / 59.0 - 0.5,
        'hour':       dates.hour / 23.0 - 0.5,
        'dayofweek':  dates.dayofweek / 6.0 - 0.5,
        'dayofmonth': (dates.day - 1) / 30.0 - 0.5,
        'dayofyear':  (dates.dayofyear - 1) / 365.0 - 0.5,
    }
    order = {
        't': ['minute', 'hour', 'dayofweek', 'dayofmonth', 'dayofyear'],
        'h': ['hour', 'dayofweek', 'dayofmonth', 'dayofyear'],
        'd': ['dayofweek', 'dayofmonth', 'dayofyear'],
    }[freq]
    return np.stack([np.asarray(feats[k], dtype=np.float64) for k in order], axis=1)

FREQ_TO_NFEAT = {'t': 5, 'h': 4, 'd': 3}


class TimeSeriesDataset(Dataset):
    def __init__(self, csv_path, split, seq_len, label_len, pred_len,
                 split_type='custom', freq='h', target='OT'):
        assert split in ('train', 'val', 'test')
        self.seq_len, self.label_len, self.pred_len = seq_len, label_len, pred_len

        df = pd.read_csv(csv_path)
        # custom 데이터는 [date, 나머지 변수..., target] 순서로 정렬 (ETT는 OT가 이미 마지막 열)
        cols = [c for c in df.columns if c not in ('date', target)]
        df = df[['date'] + cols + [target]]
        n = len(df)

        # ---- 1-1. 분할 경계 ----
        if split_type in ('ETT_hour', 'ETT_minute'):
            unit = 24 if split_type == 'ETT_hour' else 24 * 4      # 하루당 시점 수
            train_end = 12 * 30 * unit
            val_end = train_end + 4 * 30 * unit
            test_end = val_end + 4 * 30 * unit
        else:
            num_train, num_test = int(n * 0.7), int(n * 0.2)
            train_end, val_end, test_end = num_train, n - num_test, n
        starts = {'train': 0, 'val': train_end - seq_len, 'test': val_end - seq_len}
        ends = {'train': train_end, 'val': val_end, 'test': test_end}
        b1, b2 = starts[split], ends[split]

        # ---- 1-2. 정규화: 학습 구간 통계만 사용 ----
        values = df.iloc[:, 1:].values.astype(np.float64)
        train_values = values[:train_end]
        self.mean = train_values.mean(axis=0)
        self.std = train_values.std(axis=0)          # 모집단 표준편차 (sklearn StandardScaler와 동일)
        self.std[self.std == 0] = 1.0
        data = (values - self.mean) / self.std

        # ---- 1-3. 시간 특징 ----
        dates = pd.to_datetime(df['date'].values[b1:b2])
        self.data_x = data[b1:b2]
        self.data_stamp = time_features(pd.DatetimeIndex(dates), freq)

    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1

    def __getitem__(self, i):
        # ---- 1-4. 슬라이딩 윈도우 ----
        s_begin, s_end = i, i + self.seq_len                    # 인코더 입력 구간
        r_begin = s_end - self.label_len                        # 디코더 구간 시작 (입력과 48개 겹침)
        r_end = r_begin + self.label_len + self.pred_len
        x = self.data_x[s_begin:s_end]
        y = self.data_x[r_begin:r_end]
        x_mark = self.data_stamp[s_begin:s_end]
        y_mark = self.data_stamp[r_begin:r_end]
        return (torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32),
                torch.tensor(x_mark, dtype=torch.float32), torch.tensor(y_mark, dtype=torch.float32))


def make_loaders(cfg, num_workers=2):
    # 공식 코드와 같은 설정: 학습·검증은 shuffle + drop_last, 테스트는 순서대로 전부 사용
    path = os.path.join(DATA_ROOT, cfg.data_path)
    sets = {s: TimeSeriesDataset(path, s, cfg.seq_len, cfg.label_len, cfg.pred_len,
                                 cfg.split_type, cfg.freq) for s in ('train', 'val', 'test')}
    loaders = {
        'train': DataLoader(sets['train'], cfg.batch_size, shuffle=True, drop_last=True, num_workers=num_workers),
        'val':   DataLoader(sets['val'],   cfg.batch_size, shuffle=True, drop_last=True, num_workers=num_workers),
        'test':  DataLoader(sets['test'],  cfg.batch_size, shuffle=False, drop_last=False, num_workers=num_workers),
    }
    return sets, loaders

**확인**: ETTm2 기준으로 샘플 수가 공식 코드 로그와 같은 `train 34369 / val 11425 / test 11425`가 나와야 합니다.

In [ ]:
_check = {s: TimeSeriesDataset(os.path.join(DATA_ROOT, 'ETT-small/ETTm2.csv'), s, 96, 48, 96, 'ETT_minute', 't')
          for s in ('train', 'val', 'test')}
for s, d in _check.items():
    print(f'{s:5s} {len(d)}')
x, y, xm, ym = _check['train'][0]
print('x', tuple(x.shape), '| y', tuple(y.shape), '| x_mark', tuple(xm.shape), '| y_mark', tuple(ym.shape))
print('y의 앞 48개 == x의 뒤 48개:', torch.equal(y[:48], x[-48:]))

## 2. Series Decomposition 블록 (논문 식 1)

$$\mathcal{X}_t = \mathrm{AvgPool}(\mathrm{Padding}(\mathcal{X})), \qquad \mathcal{X}_s = \mathcal{X} - \mathcal{X}_t$$

- **추세(trend-cyclical)** $\mathcal{X}_t$: 이동평균으로 짧은 진동을 평탄하게 만든 장기 흐름
- **계절(seasonal)** $\mathcal{X}_s$: 원래 값에서 추세를 뺀 나머지, 즉 반복되는 진동

핵심 아이디어는 이 분해를 **전처리로 한 번만 하는 것이 아니라, 모델 내부 블록으로 여러 번 반복**한다는 점입니다 (progressive decomposition).
미래 구간은 원래 값을 모르기 때문에 미리 분해할 수 없지만, 모델 안의 은닉 표현에 분해를 반복 적용하면 예측 중간 결과에서도 추세를 점점 걸러낼 수 있습니다.

**구현 디테일**
- 이동평균 커널 크기는 25입니다 (`moving_avg=25`).
- 길이를 유지하기 위해 양 끝에 12개씩 패딩합니다. 0이 아니라 **첫 값과 마지막 값을 반복**해서 채웁니다. 0으로 채우면 양 끝의 추세가 0쪽으로 끌려 내려가기 때문입니다.
- 학습 파라미터가 없는 순수한 신호 처리 연산입니다.

In [ ]:
class MovingAvg(nn.Module):
    def __init__(self, kernel_size, stride=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):                                   # x: [B, L, D]
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)               # 첫 값 반복
        end = x[:, -1:, :].repeat(1, pad, 1)                # 마지막 값 반복
        x = torch.cat([front, x, end], dim=1)               # [B, L + 2*pad, D]
        return self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)  # AvgPool1d는 [B, D, L] 입력


class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)          # X_t
        seasonal = x - trend                # X_s
        return seasonal, trend

**확인**: 직선(추세) + 주기 24인 사인파(계절)를 더한 인공 신호를 넣으면, 두 성분이 분리되어야 합니다.

In [ ]:
t = torch.arange(200, dtype=torch.float32)
true_trend = 0.02 * t
true_season = torch.sin(2 * math.pi * t / 24)
signal = (true_trend + true_season).view(1, -1, 1)

seasonal, trend = SeriesDecomp(25)(signal)
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].plot(signal[0, :, 0]); axes[0].set_title('input = trend + seasonal')
axes[1].plot(true_trend, '--', label='true'); axes[1].plot(trend[0, :, 0], label='extracted'); axes[1].set_title('trend'); axes[1].legend()
axes[2].plot(true_season, '--', label='true'); axes[2].plot(seasonal[0, :, 0], label='extracted'); axes[2].set_title('seasonal'); axes[2].legend()
plt.tight_layout(); save_fig('01_series_decomp_synthetic'); plt.show()
print('출력 길이 유지:', signal.shape == trend.shape == seasonal.shape)

## 3. Auto-Correlation 메커니즘 (논문 3.2절)

Self-attention이 **점(point) 대 점**의 유사도를 계산한다면, Auto-Correlation은 **주기에 기반한 부분 시계열(sub-series) 단위**로 의존성을 찾고 정보를 모읍니다.

### 3-1. 주기 기반 의존성 찾기 (식 5)
$$\mathcal{R}_{\mathcal{X}\mathcal{X}}(\tau) = \lim_{L\to\infty}\frac{1}{L}\sum_{t=1}^{L}\mathcal{X}_t\mathcal{X}_{t-\tau}$$
시계열을 $\tau$만큼 밀었을 때 원래 시계열과 얼마나 비슷한지를 나타냅니다. 주기가 24라면 $\tau=24$에서 값이 크게 나옵니다.
즉 $\mathcal{R}(\tau)$가 큰 $\tau$가 곧 **주기 후보**입니다.

### 3-2. FFT로 한 번에 계산 (식 8, Wiener–Khinchin 정리)
모든 $\tau = 1, \dots, L$에 대해 직접 계산하면 $O(L^2)$입니다. 대신
$$\mathcal{S}(f) = \mathcal{F}(\mathcal{Q})\,\overline{\mathcal{F}(\mathcal{K})}, \qquad \mathcal{R}(\tau) = \mathcal{F}^{-1}(\mathcal{S}(f))$$
처럼 FFT → 켤레 곱 → 역 FFT를 하면 **모든 지연의 자기상관을 $O(L\log L)$로 한 번에** 얻습니다.
(엄밀히는 원형(circular) 상관입니다. 시계열 끝에서 밀려난 값이 앞으로 돌아오는 방식입니다.)

### 3-3. 상위 k개 지연 선택과 Time Delay Aggregation (식 6)
$$\tau_1,\dots,\tau_k = \arg\mathrm{Topk}(\mathcal{R}(\tau)),\quad k = \lfloor c \times \log L \rfloor$$
$$\mathrm{AutoCorrelation}(\mathcal{Q},\mathcal{K},\mathcal{V}) = \sum_{i=1}^{k}\mathrm{Roll}(\mathcal{V},\tau_i)\,\mathrm{SoftMax}(\mathcal{R})_i$$
- 상관이 가장 큰 $k$개의 지연을 고릅니다. $c$는 하이퍼파라미터(`factor`)입니다.
- $\mathcal{V}$를 각 지연만큼 **굴려서(roll)** 같은 위상(phase)에 있는 부분 시계열끼리 정렬한 뒤, softmax 가중치로 더합니다.
- 예: 주기가 24이면 "어제 같은 시각"의 흐름 전체를 오늘 위치로 가져와 합치는 것과 같습니다.

### 3-4. Speedup 버전 (Appendix G, Algorithm 3·4)
논문의 모든 실험은 속도를 높인 버전으로 얻은 것입니다.
- **head·채널 방향으로 상관을 평균**내서, 지연 선택을 샘플당 한 번만 합니다.
- **학습 시**: 배치 전체 평균으로 지연 $k$개를 고르고 배치의 모든 샘플에 같은 지연을 적용합니다 (`torch.roll`). 가중치(softmax)는 샘플마다 따로 계산합니다. 배치 정규화처럼 배치 단위로 안정적인 판단을 내리는 효과가 있습니다.
- **추론 시**: 샘플마다 자기 자신의 지연 $k$개를 고릅니다 (`torch.gather`).
  공식 코드의 추론 버전은 부록 Algorithm 4의 의사코드(배치 평균)와 달리 샘플별 지연을 씁니다. 여기서는 공식 코드를 따랐습니다.

### 3-5. 길이가 다른 경우 (인코더-디코더 Auto-Correlation)
디코더 쪽 $\mathcal{Q}$의 길이(48+96=144)가 인코더 쪽 $\mathcal{K},\mathcal{V}$의 길이(96)보다 길면, $\mathcal{K},\mathcal{V}$ 뒤를 0으로 채워 길이를 맞춥니다. 반대면 앞부분만 잘라 씁니다.

In [ ]:
class AutoCorrelation(nn.Module):
    def __init__(self, factor=1):
        super().__init__()
        self.factor = factor          # 논문의 c
        self.record = False           # True면 지연별 상관을 저장 (섹션 11의 주기 분석용)
        self.last_corr = None

    def time_delay_agg_training(self, values, corr):
        # values, corr: [B, H, C, L]
        B, H, C, L = values.shape
        top_k = int(self.factor * math.log(L))
        mean_value = corr.mean(dim=1).mean(dim=1)                     # head·채널 평균 → [B, L]
        index = torch.topk(mean_value.mean(dim=0), top_k, dim=-1)[1]  # 배치 공통 지연 k개 → [k]
        weights = mean_value[:, index]                                # 샘플별 해당 지연의 상관 → [B, k]
        tmp_corr = torch.softmax(weights, dim=-1)
        out = torch.zeros_like(values)
        for i in range(top_k):
            pattern = torch.roll(values, shifts=-int(index[i]), dims=-1)   # 지연만큼 당겨서 정렬
            out = out + pattern * tmp_corr[:, i].view(B, 1, 1, 1)
        return out

    def time_delay_agg_inference(self, values, corr):
        B, H, C, L = values.shape
        top_k = int(self.factor * math.log(L))
        mean_value = corr.mean(dim=1).mean(dim=1)                     # [B, L]
        weights, delay = torch.topk(mean_value, top_k, dim=-1)        # 샘플별 지연 k개 → [B, k]
        tmp_corr = torch.softmax(weights, dim=-1)
        base = torch.arange(L, device=values.device).view(1, 1, 1, L)
        out = torch.zeros_like(values)
        for i in range(top_k):
            # 위치 t에 values[(t + τ) mod L]을 가져옴 = 샘플마다 다른 τ로 roll(-τ)한 것과 같음
            idx = (base + delay[:, i].view(B, 1, 1, 1)) % L
            pattern = torch.gather(values, -1, idx.expand(B, H, C, L))
            out = out + pattern * tmp_corr[:, i].view(B, 1, 1, 1)
        return out

    def forward(self, queries, keys, values):
        # queries: [B, L, H, E], keys/values: [B, S, H, E]
        B, L, H, E = queries.shape
        _, S, _, _ = values.shape
        if L > S:   # 3-5. 길이 맞추기
            zeros = torch.zeros_like(queries[:, :(L - S), :, :])
            values = torch.cat([values, zeros], dim=1)
            keys = torch.cat([keys, zeros], dim=1)
        else:
            values, keys = values[:, :L], keys[:, :L]

        # 3-2. FFT로 모든 지연의 자기상관 계산 (시간축을 마지막 차원으로)
        q_fft = torch.fft.rfft(queries.permute(0, 2, 3, 1).contiguous(), dim=-1)
        k_fft = torch.fft.rfft(keys.permute(0, 2, 3, 1).contiguous(), dim=-1)
        corr = torch.fft.irfft(q_fft * torch.conj(k_fft), n=L, dim=-1)    # [B, H, E, L]
        if self.record:
            self.last_corr = corr.mean(dim=1).mean(dim=1).detach().cpu()  # [B, L]

        # 3-3, 3-4. Time Delay Aggregation
        v = values.permute(0, 2, 3, 1).contiguous()
        agg = self.time_delay_agg_training(v, corr) if self.training else self.time_delay_agg_inference(v, corr)
        return agg.permute(0, 3, 1, 2).contiguous()                       # [B, L, H, E]


class AutoCorrelationLayer(nn.Module):
    # Multi-head 버전 (논문 식 7): Q, K, V 선형 사영 → head로 나눔 → Auto-Correlation → 합쳐서 출력 사영
    # self-attention의 AttentionLayer와 골격이 같으므로 그대로 갈아 끼울 수 있음
    def __init__(self, correlation, d_model, n_heads):
        super().__init__()
        d_head = d_model // n_heads
        self.inner_correlation = correlation
        self.query_projection = nn.Linear(d_model, d_head * n_heads)
        self.key_projection = nn.Linear(d_model, d_head * n_heads)
        self.value_projection = nn.Linear(d_model, d_head * n_heads)
        self.out_projection = nn.Linear(d_head * n_heads, d_model)
        self.n_heads = n_heads

    def forward(self, queries, keys, values):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads
        q = self.query_projection(queries).view(B, L, H, -1)
        k = self.key_projection(keys).view(B, S, H, -1)
        v = self.value_projection(values).view(B, S, H, -1)
        out = self.inner_correlation(q, k, v).view(B, L, -1)
        return self.out_projection(out)

**확인**: 주기 24인 신호에서 FFT로 계산한 자기상관이 직접 계산한 값과 같은지, 그리고 상관이 큰 지연이 24의 배수 근처로 나오는지 확인합니다.
(지연 1처럼 아주 짧은 지연도 나올 수 있습니다. 바로 이웃한 시점끼리는 값이 비슷하기 때문입니다. 실제 모델은 이렇게 고른 여러 지연을 softmax 가중치로 섞어서 사용합니다.)

In [ ]:
L = 96
t = torch.arange(L, dtype=torch.float32)
sig = torch.sin(2 * math.pi * t / 24) + 0.1 * torch.randn(L)

# 직접 계산한 원형 자기상관 R(τ) = Σ_t x[t+τ] x[t]  vs  FFT로 계산한 값
direct = torch.stack([(torch.roll(sig, -tau) * sig).sum() for tau in range(L)])
fft_val = torch.fft.irfft(torch.fft.rfft(sig) * torch.conj(torch.fft.rfft(sig)), n=L)
print('직접 계산과 FFT 결과 일치:', torch.allclose(direct, fft_val, atol=1e-3))

top = torch.topk(fft_val[1:L // 2], 4).indices + 1   # τ=0(자기 자신)과 대칭인 뒤쪽 절반 제외
print('상관이 큰 지연 상위 4개:', sorted(top.tolist()))

plt.figure(figsize=(10, 3))
plt.bar(range(L), fft_val.numpy()); plt.xlabel('lag τ'); plt.title('Autocorrelation R(τ) of a period-24 signal')
save_fig('02_autocorrelation_synthetic'); plt.show()

## 4. 임베딩 (Appendix G.2)

입력 값과 시간 특징을 각각 `d_model`(512) 차원으로 바꿔 더합니다.

- **값 임베딩 (TokenEmbedding)**: 커널 3짜리 1D 합성곱 (원형 패딩, bias 없음). 바로 이웃한 시점 정보가 함께 섞입니다.
- **시간 특징 임베딩 (TimeFeatureEmbedding)**: 시간 특징 4~5개를 선형 변환 (bias 없음).
- **위치 임베딩은 쓰지 않습니다.** 일반 Transformer는 self-attention이 순서를 모르기 때문에 위치 임베딩이 필요하지만, Auto-Correlation은 시간 지연(roll) 자체가 순서 정보를 담고 있기 때문입니다.

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super().__init__()
        self.tokenConv = nn.Conv1d(c_in, d_model, kernel_size=3, padding=1,
                                   padding_mode='circular', bias=False)
        nn.init.kaiming_normal_(self.tokenConv.weight, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, x):                                   # [B, L, c_in] → [B, L, d_model]
        return self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2)


class TimeFeatureEmbedding(nn.Module):
    def __init__(self, d_model, freq='h'):
        super().__init__()
        self.embed = nn.Linear(FREQ_TO_NFEAT[freq], d_model, bias=False)

    def forward(self, x_mark):
        return self.embed(x_mark)


class DataEmbeddingWoPos(nn.Module):
    def __init__(self, c_in, d_model, freq='h', dropout=0.05):
        super().__init__()
        self.value_embedding = TokenEmbedding(c_in, d_model)
        self.temporal_embedding = TimeFeatureEmbedding(d_model, freq)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, x_mark):
        return self.dropout(self.value_embedding(x) + self.temporal_embedding(x_mark))

## 5. 인코더와 디코더 (논문 식 3, 4)

### 5-1. 계절 성분 전용 LayerNorm (`MyLayerNorm`)
일반 LayerNorm을 적용한 뒤 **시간축 평균을 한 번 더 뺍니다.** 계절 성분은 평균이 0인 순수한 진동이어야 한다는 전제를 유지하기 위한 장치입니다.

### 5-2. 인코더 층 (식 3)
$$\mathcal{S}^{l,1}_{en}, \_ = \mathrm{SeriesDecomp}\big(\mathrm{AutoCorrelation}(\mathcal{X}^{l-1}_{en}) + \mathcal{X}^{l-1}_{en}\big)$$
$$\mathcal{S}^{l,2}_{en}, \_ = \mathrm{SeriesDecomp}\big(\mathrm{FeedForward}(\mathcal{S}^{l,1}_{en}) + \mathcal{S}^{l,1}_{en}\big)$$
- 분해할 때마다 **추세(`_`)는 버리고 계절 성분만** 다음으로 넘깁니다. 인코더는 계절 패턴 모델링에 집중합니다.
- Feed Forward는 커널 크기 1인 Conv1d 두 개(512 → 2048 → 512, bias 없음)로, 시점별 MLP와 같습니다.

### 5-3. 디코더 층 (식 4)
$$\mathcal{S}^{l,1}_{de}, \mathcal{T}^{l,1}_{de} = \mathrm{SeriesDecomp}\big(\mathrm{AutoCorrelation}(\mathcal{X}^{l-1}_{de}) + \mathcal{X}^{l-1}_{de}\big)$$
$$\mathcal{S}^{l,2}_{de}, \mathcal{T}^{l,2}_{de} = \mathrm{SeriesDecomp}\big(\mathrm{AutoCorrelation}(\mathcal{S}^{l,1}_{de}, \mathcal{X}^{N}_{en}) + \mathcal{S}^{l,1}_{de}\big)$$
$$\mathcal{S}^{l,3}_{de}, \mathcal{T}^{l,3}_{de} = \mathrm{SeriesDecomp}\big(\mathrm{FeedForward}(\mathcal{S}^{l,2}_{de}) + \mathcal{S}^{l,2}_{de}\big)$$
$$\mathcal{T}^{l}_{de} = \mathcal{T}^{l-1}_{de} + \mathcal{W}_{l,1}\mathcal{T}^{l,1}_{de} + \mathcal{W}_{l,2}\mathcal{T}^{l,2}_{de} + \mathcal{W}_{l,3}\mathcal{T}^{l,3}_{de}$$
- 첫 번째 Auto-Correlation은 디코더 자기 자신(inner), 두 번째는 **인코더 출력과의 교차**(encoder-decoder)입니다. 교차에서 $\mathcal{Q}$는 디코더, $\mathcal{K},\mathcal{V}$는 인코더에서 옵니다.
- 이번에는 분해에서 나온 **추세 세 개를 버리지 않고 누적**합니다. 이것이 Figure 1 아래쪽의 "Trend-cyclical" 경로입니다.

**논문 식과 공식 코드의 차이**: 식 4는 사영 $\mathcal{W}_{l,1}, \mathcal{W}_{l,2}, \mathcal{W}_{l,3}$를 따로 두는 것처럼 보이지만, 공식 코드는
추세 세 개를 **먼저 더한 뒤 사영 하나**(커널 3짜리 Conv1d, 원형 패딩)만 적용합니다. 여기서는 공식 코드를 따랐습니다.
이 사영은 `d_model`(512) 차원의 추세를 원래 변수 수(`c_out`, ETT는 7)로 줄이는 역할도 합니다.

In [ ]:
class MyLayerNorm(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layernorm = nn.LayerNorm(channels)

    def forward(self, x):
        x_hat = self.layernorm(x)
        return x_hat - x_hat.mean(dim=1, keepdim=True)     # 시간축 평균 제거


class EncoderLayer(nn.Module):
    def __init__(self, attention, d_model, d_ff=2048, moving_avg=25, dropout=0.05, activation='gelu'):
        super().__init__()
        self.attention = attention
        self.conv1 = nn.Conv1d(d_model, d_ff, kernel_size=1, bias=False)
        self.conv2 = nn.Conv1d(d_ff, d_model, kernel_size=1, bias=False)
        self.decomp1 = SeriesDecomp(moving_avg)
        self.decomp2 = SeriesDecomp(moving_avg)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == 'relu' else F.gelu

    def forward(self, x):
        x = x + self.dropout(self.attention(x, x, x))
        x, _ = self.decomp1(x)                                          # S^{l,1}, 추세 버림
        y = self.dropout(self.activation(self.conv1(x.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))
        x, _ = self.decomp2(x + y)                                      # S^{l,2}, 추세 버림
        return x


class Encoder(nn.Module):
    def __init__(self, attn_layers, norm_layer=None):
        super().__init__()
        self.attn_layers = nn.ModuleList(attn_layers)
        self.norm = norm_layer

    def forward(self, x):
        for layer in self.attn_layers:
            x = layer(x)
        return self.norm(x) if self.norm is not None else x


class DecoderLayer(nn.Module):
    def __init__(self, self_attention, cross_attention, d_model, c_out, d_ff=2048,
                 moving_avg=25, dropout=0.05, activation='gelu'):
        super().__init__()
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.conv1 = nn.Conv1d(d_model, d_ff, kernel_size=1, bias=False)
        self.conv2 = nn.Conv1d(d_ff, d_model, kernel_size=1, bias=False)
        self.decomp1 = SeriesDecomp(moving_avg)
        self.decomp2 = SeriesDecomp(moving_avg)
        self.decomp3 = SeriesDecomp(moving_avg)
        self.dropout = nn.Dropout(dropout)
        # 추세 사영: d_model → c_out (식 4의 W를 하나로 구현)
        self.projection = nn.Conv1d(d_model, c_out, kernel_size=3, stride=1, padding=1,
                                    padding_mode='circular', bias=False)
        self.activation = F.relu if activation == 'relu' else F.gelu

    def forward(self, x, cross):
        x = x + self.dropout(self.self_attention(x, x, x))             # inner Auto-Correlation
        x, trend1 = self.decomp1(x)
        x = x + self.dropout(self.cross_attention(x, cross, cross))     # encoder-decoder Auto-Correlation
        x, trend2 = self.decomp2(x)
        y = self.dropout(self.activation(self.conv1(x.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))
        x, trend3 = self.decomp3(x + y)
        residual_trend = trend1 + trend2 + trend3
        residual_trend = self.projection(residual_trend.permute(0, 2, 1)).transpose(1, 2)
        return x, residual_trend


class Decoder(nn.Module):
    def __init__(self, layers, norm_layer=None, projection=None):
        super().__init__()
        self.layers = nn.ModuleList(layers)
        self.norm = norm_layer
        self.projection = projection

    def forward(self, x, cross, trend):
        for layer in self.layers:
            x, residual_trend = layer(x, cross)
            trend = trend + residual_trend                              # 추세 누적
        if self.norm is not None:
            x = self.norm(x)
        if self.projection is not None:
            x = self.projection(x)                                      # 계절 성분 d_model → c_out
        return x, trend

## 6. 전체 모델 (논문 식 2, Figure 1)

### 디코더 입력 초기화 (식 2)
$$\mathcal{X}_{des} = \mathrm{Concat}(\mathcal{X}_{ens}, \mathcal{X}_0), \qquad \mathcal{X}_{det} = \mathrm{Concat}(\mathcal{X}_{ent}, \mathcal{X}_{Mean})$$
- **계절 초기값**: 최근 48개 구간의 계절 성분 + 예측 구간 96칸은 **0**
- **추세 초기값**: 최근 48개 구간의 추세 성분 + 예측 구간 96칸은 **입력 96개의 평균**
- 디코더는 계절 초기값을 임베딩해서 처리하고, 추세 초기값에는 층마다 추출한 추세를 계속 더합니다.

**최종 예측** = 계절 출력(사영) + 누적된 추세, 그중 마지막 96칸 (`pred_len`)

**논문과 공식 코드의 차이**: 식 2는 입력의 **뒤쪽 절반만** 분해한다고 쓰지만, 공식 코드는 입력 96개 **전체를 분해한 뒤** 마지막 48개를 잘라 씁니다.
이동평균은 양 끝에서 값이 달라지므로 두 방식의 결과가 조금 다릅니다. 여기서는 공식 코드를 따랐습니다.

### 기본 설정 (공식 `run.py` 기본값)
`d_model=512, n_heads=8, e_layers=2, d_layers=1, d_ff=2048, moving_avg=25, dropout=0.05, activation=gelu`

In [ ]:
class Autoformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.label_len, self.pred_len = cfg.label_len, cfg.pred_len
        self.decomp = SeriesDecomp(cfg.moving_avg)
        self.enc_embedding = DataEmbeddingWoPos(cfg.enc_in, cfg.d_model, cfg.freq, cfg.dropout)
        self.dec_embedding = DataEmbeddingWoPos(cfg.dec_in, cfg.d_model, cfg.freq, cfg.dropout)
        self.encoder = Encoder(
            [EncoderLayer(AutoCorrelationLayer(AutoCorrelation(cfg.factor), cfg.d_model, cfg.n_heads),
                          cfg.d_model, cfg.d_ff, cfg.moving_avg, cfg.dropout, cfg.activation)
             for _ in range(cfg.e_layers)],
            norm_layer=MyLayerNorm(cfg.d_model))
        self.decoder = Decoder(
            [DecoderLayer(AutoCorrelationLayer(AutoCorrelation(cfg.factor), cfg.d_model, cfg.n_heads),
                          AutoCorrelationLayer(AutoCorrelation(cfg.factor), cfg.d_model, cfg.n_heads),
                          cfg.d_model, cfg.c_out, cfg.d_ff, cfg.moving_avg, cfg.dropout, cfg.activation)
             for _ in range(cfg.d_layers)],
            norm_layer=MyLayerNorm(cfg.d_model),
            projection=nn.Linear(cfg.d_model, cfg.c_out, bias=True))

    def forward(self, x_enc, x_mark_enc, x_mark_dec, return_parts=False):
        # x_enc: [B, 96, D], x_mark_enc: [B, 96, n_time], x_mark_dec: [B, 48+96, n_time]
        B, _, D = x_enc.shape
        # ---- 식 2: 디코더 입력 초기화 ----
        mean = x_enc.mean(dim=1, keepdim=True).repeat(1, self.pred_len, 1)
        zeros = torch.zeros(B, self.pred_len, D, device=x_enc.device)
        seasonal_init, trend_init = self.decomp(x_enc)
        trend_init = torch.cat([trend_init[:, -self.label_len:, :], mean], dim=1)       # [B, 144, D]
        seasonal_init = torch.cat([seasonal_init[:, -self.label_len:, :], zeros], dim=1)  # [B, 144, D]
        # ---- 인코더 ----
        enc_out = self.encoder(self.enc_embedding(x_enc, x_mark_enc))                  # [B, 96, d_model]
        # ---- 디코더 ----
        dec_in = self.dec_embedding(seasonal_init, x_mark_dec)                         # [B, 144, d_model]
        seasonal_part, trend_part = self.decoder(dec_in, enc_out, trend=trend_init)    # 둘 다 [B, 144, D]
        out = trend_part + seasonal_part
        if return_parts:
            return out[:, -self.pred_len:], seasonal_part[:, -self.pred_len:], trend_part[:, -self.pred_len:]
        return out[:, -self.pred_len:]                                                  # [B, 96, D]


class Config:
    # 공식 run.py 기본값 + 데이터셋별 설정을 담는 간단한 설정 객체
    def __init__(self, **kw):
        defaults = dict(seq_len=96, label_len=48, pred_len=96, enc_in=7, dec_in=7, c_out=7,
                        d_model=512, n_heads=8, e_layers=2, d_layers=1, d_ff=2048, moving_avg=25,
                        factor=1, dropout=0.05, activation='gelu', freq='h',
                        batch_size=32, learning_rate=1e-4, train_epochs=10, patience=3,
                        data_path='ETT-small/ETTm2.csv', split_type='ETT_minute')
        defaults.update(kw)
        self.__dict__.update(defaults)

    def __repr__(self):
        return 'Config(' + ', '.join(f'{k}={v}' for k, v in self.__dict__.items()) + ')'

**확인**: 임의 입력으로 출력 크기와 파라미터 수를 확인합니다.

In [ ]:
_cfg = Config(freq='t')
_m = Autoformer(_cfg)
_x, _y, _xm, _ym = [t.unsqueeze(0).repeat(4, 1, 1) for t in _check['train'][0]]
print('출력 shape:', tuple(_m(_x, _xm, _ym).shape), '(기대값: (4, 96, 7))')
print('학습 파라미터 수: {:,}'.format(sum(p.numel() for p in _m.parameters() if p.requires_grad)))

## 7. (선택) 공식 코드와 출력 일치 검증

직접 구현이 공식 구현과 **수학적으로 같은지** 확인하는 섹션입니다. 모듈 이름을 공식 코드와 똑같이 지었기 때문에, 공식 모델의 가중치를 그대로 불러올 수 있습니다.
**같은 가중치 + 같은 입력 → 같은 출력**이 나오면 구현이 일치한다는 강한 근거가 됩니다. 발표의 "구현 검증" 근거로 쓸 수 있습니다.

- 공식 저장소가 `OFFICIAL_DIR`에 clone되어 있어야 합니다. 없으면 이 섹션은 건너뜁니다.
- 학습 모드 비교는 dropout을 0으로 두고 합니다 (dropout은 무작위라 매번 달라지기 때문).
- 데이터도 공식 로더와 같은 값이 나오는지 비교합니다.

In [ ]:
from types import SimpleNamespace

def verify_against_official():
    if not os.path.isdir(os.path.join(OFFICIAL_DIR, 'models')):
        print('공식 코드가 없어서 검증을 건너뜁니다:', OFFICIAL_DIR)
        return
    if OFFICIAL_DIR not in sys.path:
        sys.path.insert(0, OFFICIAL_DIR)
    import importlib
    official = importlib.import_module('models.Autoformer')

    for freq, dropout in [('t', 0.0), ('h', 0.0)]:
        kw = dict(seq_len=96, label_len=48, pred_len=96, enc_in=7, dec_in=7, c_out=7, d_model=64, n_heads=8,
                  e_layers=2, d_layers=1, d_ff=128, moving_avg=25, factor=1, dropout=dropout,
                  activation='gelu', freq=freq, embed='timeF', output_attention=False)
        torch.manual_seed(0)
        ref = official.Model(SimpleNamespace(**kw)).float()
        mine = Autoformer(Config(**kw)).float()
        missing, unexpected = mine.load_state_dict(ref.state_dict(), strict=False)
        print(f'[freq={freq}] 불러오지 못한 내 파라미터: {missing}')
        print(f'[freq={freq}] 공식에만 있는 항목: {unexpected}  (위치 임베딩 버퍼 pe는 사용하지 않으므로 정상)')

        n_time = FREQ_TO_NFEAT[freq]
        x = torch.randn(8, 96, 7); xm = torch.rand(8, 96, n_time) - 0.5
        ym = torch.rand(8, 144, n_time) - 0.5
        dec_inp = torch.zeros(8, 144, 7)   # 공식 forward 형식 맞춤용 (Autoformer는 값 자체는 사용하지 않음)
        for mode in ('eval', 'train'):
            ref.train(mode == 'train'); mine.train(mode == 'train')
            with torch.no_grad():
                a = ref(x, xm, dec_inp, ym); b = mine(x, xm, ym)
            print(f'  {mode:5s} 모드 최대 오차: {(a - b).abs().max().item():.2e}  일치: {torch.allclose(a, b, atol=1e-5)}')

    # 데이터 로더 비교 (freq='h'는 모든 pandas 버전에서 공식 코드가 동작)
    from data_provider.data_loader import Dataset_ETT_minute
    for split in ('train', 'val', 'test'):
        o = Dataset_ETT_minute(os.path.join(DATA_ROOT, 'ETT-small'), flag=split, size=[96, 48, 96],
                               features='M', data_path='ETTm2.csv', timeenc=1, freq='h')
        m = TimeSeriesDataset(os.path.join(DATA_ROOT, 'ETT-small/ETTm2.csv'), split, 96, 48, 96, 'ETT_minute', 'h')
        same = (len(o) == len(m) and np.allclose(o.data_x, m.data_x, atol=1e-6)
                and np.allclose(o.data_stamp, m.data_stamp, atol=1e-9))
        print(f'데이터 [{split}] 샘플 수 {len(o)} vs {len(m)}, 값 일치: {same}')

verify_against_official()

## 8. 학습 · 평가 루프 (논문 4장 Implementation details + 공식 코드)

| 항목 | 설정 |
|---|---|
| 손실 함수 | MSE (L2) |
| 옵티마이저 | Adam, 초기 학습률 $10^{-4}$ |
| 배치 크기 | 32 |
| 최대 epoch | 10 |
| Early stopping | 검증 손실이 **3번 연속** 개선되지 않으면 중단, 검증 손실이 가장 낮았던 가중치로 테스트 |
| 학습률 스케줄 | epoch마다 절반 (1~2 epoch은 $10^{-4}$, 3 epoch은 $5\times10^{-5}$, ...) — 논문에는 없고 공식 코드(`lradj=type1`)에 있음 |
| 평가 지표 | 테스트 구간 전체 예측의 MSE, MAE (정규화된 값 기준) |
| 반복 | 같은 설정을 3번 반복해 평균과 표준편차 (논문 Table 10) |

**학습률 스케줄 상세**: 공식 코드는 epoch $e$가 끝날 때마다 학습률을 $10^{-4}\times 0.5^{\,e-1}$로 설정합니다. 그래서 1 epoch이 끝난 뒤에는 그대로 $10^{-4}$이고, 2 epoch이 끝난 뒤부터 절반씩 줄어듭니다.

**디코더 입력**: 공식 코드는 `[실제 y 앞 48개, 0 × 96]`을 디코더 입력으로 넘기지만, Autoformer의 `forward`는 이 값을 쓰지 않고 시간 특징(`y_mark`)만 사용합니다. 그래서 여기서는 `x_mark_dec`만 넘깁니다.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience, self.counter = patience, 0
        self.best_loss, self.best_state, self.best_epoch = None, None, 0
        self.early_stop = False

    def step(self, val_loss, model, epoch):
        if self.best_loss is None or val_loss <= self.best_loss:     # 개선(같아도 개선으로 봄, 공식과 동일)
            self.best_loss, self.best_epoch, self.counter = val_loss, epoch, 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f'    검증 손실 개선 → 가중치 저장 (epoch {epoch})')
        else:
            self.counter += 1
            print(f'    EarlyStopping counter: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True


def adjust_learning_rate(optimizer, epoch, base_lr):
    lr = base_lr * (0.5 ** (epoch - 1))
    for g in optimizer.param_groups:
        g['lr'] = lr
    return lr


def to_device(batch):
    return [b.float().to(DEVICE) for b in batch]


@torch.no_grad()
def evaluate_loss(model, loader, pred_len, max_batches=None):
    # 공식 vali(): 배치별 MSE의 단순 평균
    model.eval(); losses = []
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y, xm, ym = to_device(batch)
        out = model(x, xm, ym)
        losses.append(F.mse_loss(out.cpu(), y[:, -pred_len:].cpu()).item())
    model.train()
    return float(np.average(losses))


@torch.no_grad()
def predict(model, loader, pred_len, max_batches=None):
    model.eval(); preds, trues = [], []
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y, xm, ym = to_device(batch)
        preds.append(model(x, xm, ym).cpu().numpy())
        trues.append(y[:, -pred_len:].cpu().numpy())
    return np.concatenate(preds), np.concatenate(trues)


def train_and_test(cfg, loaders, max_steps=None, max_eval_batches=None):
    model = Autoformer(cfg).float().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    criterion = nn.MSELoss()
    early = EarlyStopping(cfg.patience)
    t0 = time.time()

    for epoch in range(1, cfg.train_epochs + 1):
        model.train(); losses = []; te = time.time()
        for step, batch in enumerate(loaders['train']):
            if max_steps and step >= max_steps:
                break
            x, y, xm, ym = to_device(batch)
            optimizer.zero_grad()
            out = model(x, xm, ym)
            loss = criterion(out, y[:, -cfg.pred_len:])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            if (step + 1) % 300 == 0:
                print(f'    step {step + 1}/{len(loaders["train"])} | loss {loss.item():.4f}')
        val_loss = evaluate_loss(model, loaders['val'], cfg.pred_len, max_eval_batches)
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch} | lr {lr_now:.2e} | train {np.mean(losses):.5f} | val {val_loss:.5f} | {time.time() - te:.1f}s')
        early.step(val_loss, model, epoch)
        if early.early_stop:
            print('  Early stopping')
            break
        adjust_learning_rate(optimizer, epoch + 1, cfg.learning_rate)

    model.load_state_dict(early.best_state)
    preds, trues = predict(model, loaders['test'], cfg.pred_len, max_eval_batches)
    mse = float(np.mean((preds - trues) ** 2))
    mae = float(np.mean(np.abs(preds - trues)))
    info = dict(best_epoch=early.best_epoch, epochs_ran=epoch, train_time_sec=round(time.time() - t0, 1))
    return model, mse, mae, info

## 9. 실험 실행

### 데이터셋별 설정 (공식 `scripts/` 기준)

| 데이터셋 | 변수 수 | factor (c) | freq | 특이 사항 |
|---|---|---|---|---|
| ETTm2 | 7 | 1 | `t` | |
| Electricity | 321 | 3 | `h` | |
| Exchange | 8 | 3 | `h` | 예측 192, 336은 `train_epochs=1` |
| Traffic | 862 | 3 | `h` | `train_epochs=3` |
| Weather | 21 | 3 | `h` | 예측 96은 `train_epochs=2` |
| ILI | 7 | 3 | `h` | 입력 36, `label_len=18`, 예측 24/36/48/60 |

(공식 ETTm2 스크립트의 예측 길이는 24/48/96/288/672이고, 본문 Table 1의 96/192/336/720과 다릅니다. 여기서는 논문 Table 1·10과 비교하기 위해 96/192/336/720을 사용합니다.)

### 결과 저장과 이어서 실행하기
- 실험 한 번이 끝날 때마다 `OUT_DIR/results.csv`에 한 줄씩 저장됩니다.
- **이미 저장된 (데이터셋, 예측 길이, 반복 번호) 조합은 건너뜁니다.** Colab 연결이 끊겨도 같은 셀을 다시 실행하면 끝난 부분은 건너뛰고 이어서 진행합니다.
- 첫 번째 반복(run 0)의 모델은 섹션 11 시각화를 위해 `OUT_DIR/checkpoints/`에 저장됩니다.

In [ ]:
DATASETS = {
    'ETTm2':       dict(data_path='ETT-small/ETTm2.csv', split_type='ETT_minute', freq='t', enc_in=7, factor=1,
                        pred_lens=[96, 192, 336, 720]),
    'Electricity': dict(data_path='electricity/electricity.csv', split_type='custom', freq='h', enc_in=321, factor=3,
                        pred_lens=[96, 192, 336, 720]),
    'Exchange':    dict(data_path='exchange_rate/exchange_rate.csv', split_type='custom', freq='h', enc_in=8, factor=3,
                        pred_lens=[96, 192, 336, 720], epochs_by_pred={192: 1, 336: 1}),
    'Traffic':     dict(data_path='traffic/traffic.csv', split_type='custom', freq='h', enc_in=862, factor=3,
                        pred_lens=[96, 192, 336, 720], train_epochs=3),
    'Weather':     dict(data_path='weather/weather.csv', split_type='custom', freq='h', enc_in=21, factor=3,
                        pred_lens=[96, 192, 336, 720], epochs_by_pred={96: 2}),
    'ILI':         dict(data_path='illness/national_illness.csv', split_type='custom', freq='h', enc_in=7, factor=3,
                        seq_len=36, label_len=18, pred_lens=[24, 36, 48, 60]),
}

RESULTS_CSV = os.path.join(OUT_DIR, 'results_quick.csv' if QUICK_TEST else 'results.csv')
CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints_quick' if QUICK_TEST else 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)


def build_config(name, pred_len):
    d = dict(DATASETS[name])
    d.pop('pred_lens')
    epochs_by_pred = d.pop('epochs_by_pred', {})
    if pred_len in epochs_by_pred:
        d['train_epochs'] = epochs_by_pred[pred_len]
    d.update(pred_len=pred_len, dec_in=d['enc_in'], c_out=d['enc_in'])
    if QUICK_TEST:
        d['train_epochs'] = 1
    return Config(**d)


def done_runs():
    if not os.path.exists(RESULTS_CSV):
        return set()
    df = pd.read_csv(RESULTS_CSV)
    return set(zip(df.dataset, df.pred_len, df.run))


def run_experiment(name, pred_lens=None, itr=3, num_workers=2):
    pred_lens = pred_lens or DATASETS[name]['pred_lens']
    if QUICK_TEST:
        pred_lens, itr = pred_lens[:1], 1
    for pred_len in pred_lens:
        cfg = build_config(name, pred_len)
        todo = [r for r in range(itr) if (name, pred_len, r) not in done_runs()]
        if not todo:
            print(f'[{name} {cfg.seq_len}→{pred_len}] 이미 완료되어 건너뜀'); continue
        set_seed(2021)                       # 공식 코드처럼 예측 길이마다 시드 2021로 시작
        _, loaders = make_loaders(cfg, num_workers)
        for r in range(itr):
            if r not in todo:
                continue
            print(f'===== [{name}] input-{cfg.seq_len}-predict-{pred_len} | run {r + 1}/{itr} =====')
            model, mse, mae, info = train_and_test(
                cfg, loaders,
                max_steps=20 if QUICK_TEST else None,
                max_eval_batches=5 if QUICK_TEST else None)
            print(f'  >>> TEST  MSE {mse:.4f} | MAE {mae:.4f}\n')
            row = dict(dataset=name, pred_len=pred_len, run=r, mse=mse, mae=mae, **info,
                       finished_at=datetime.datetime.now().isoformat(timespec='seconds'))
            pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', index=False,
                                       header=not os.path.exists(RESULTS_CSV))
            if r == 0:
                torch.save(model.state_dict(), os.path.join(CKPT_DIR, f'{name}_{pred_len}.pth'))
            del model
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()

### 실행
먼저 ETTm2만 돌립니다. 다른 데이터셋은 아래 줄의 주석(`#`)을 지우고 실행하세요.
Electricity와 Traffic은 변수가 많아서 시간이 오래 걸립니다.

In [ ]:
run_experiment('ETTm2')
# run_experiment('Exchange')
# run_experiment('ILI')
# run_experiment('Weather')
# run_experiment('Electricity')
# run_experiment('Traffic')

## 10. 결과 정리: 논문 Table 10과 비교

3번 반복한 결과의 평균 ± 표준편차를 논문 Table 10(Autoformer)과 나란히 놓습니다.
- `diff`는 (내 결과 평균 − 논문 평균)입니다. 음수면 논문보다 좋은 결과입니다.
- `within_paper_std`는 내 평균이 논문의 평균 ± 표준편차 범위 안에 드는지 표시합니다.
- 표준편차는 표본 표준편차(ddof=1)입니다. 논문은 계산 방식을 밝히지 않았습니다.

In [ ]:
# 논문 Table 10 (Autoformer, 다변량): (MSE 평균, MSE 표준편차, MAE 평균, MAE 표준편차)
PAPER = {
    'ETTm2':       {96: (0.255, .020, 0.339, .020), 192: (0.281, .027, 0.340, .025),
                    336: (0.339, .018, 0.372, .015), 720: (0.422, .015, 0.419, .010)},
    'Electricity': {96: (0.201, .003, 0.317, .004), 192: (0.222, .003, 0.334, .004),
                    336: (0.231, .006, 0.338, .004), 720: (0.254, .007, 0.361, .008)},
    'Exchange':    {96: (0.197, .019, 0.323, .012), 192: (0.300, .020, 0.369, .016),
                    336: (0.509, .041, 0.524, .016), 720: (1.447, .084, 0.941, .028)},
    'Traffic':     {96: (0.613, .028, 0.388, .012), 192: (0.616, .042, 0.382, .020),
                    336: (0.622, .016, 0.337, .011), 720: (0.660, .025, 0.408, .015)},
    'Weather':     {96: (0.266, .007, 0.336, .006), 192: (0.307, .024, 0.367, .022),
                    336: (0.359, .035, 0.395, .031), 720: (0.419, .017, 0.428, .014)},
    'ILI':         {24: (3.483, .107, 1.287, .018), 36: (3.103, .139, 1.148, .025),
                    48: (2.669, .151, 1.085, .037), 60: (2.770, .085, 1.125, .019)},
}

def summarize(csv_path=RESULTS_CSV):
    if not os.path.exists(csv_path):
        print('결과 파일이 아직 없습니다:', csv_path); return None
    df = pd.read_csv(csv_path)
    g = df.groupby(['dataset', 'pred_len']).agg(
        runs=('run', 'count'), mse_mean=('mse', 'mean'), mse_std=('mse', 'std'),
        mae_mean=('mae', 'mean'), mae_std=('mae', 'std')).reset_index()
    rows = []
    for _, r in g.iterrows():
        p = PAPER.get(r.dataset, {}).get(r.pred_len)
        row = {'dataset': r.dataset, 'pred_len': r.pred_len, 'runs': r.runs,
               'my MSE': f'{r.mse_mean:.3f} ± {0 if pd.isna(r.mse_std) else r.mse_std:.3f}',
               'paper MSE': f'{p[0]:.3f} ± {p[1]:.3f}' if p else '-',
               'MSE diff': round(r.mse_mean - p[0], 3) if p else None,
               'my MAE': f'{r.mae_mean:.3f} ± {0 if pd.isna(r.mae_std) else r.mae_std:.3f}',
               'paper MAE': f'{p[2]:.3f} ± {p[3]:.3f}' if p else '-',
               'MAE diff': round(r.mae_mean - p[2], 3) if p else None,
               'within_paper_std (MSE)': (abs(r.mse_mean - p[0]) <= p[1]) if p else None}
        rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT_DIR, 'summary_vs_paper.csv'), index=False)
    return out

summarize()

## 11. 시각화

저장된 첫 번째 반복(run 0) 모델로 그립니다. 모든 그래프는 `OUT_DIR/figures/`에 PNG로 저장되고, 마지막에 `OUT_DIR/figures.zip`으로 묶입니다.

### 11-1. 예측 결과 (논문 Figure 8 형식)
입력 96개 뒤에 실제값(파랑)과 예측값(주황)을 이어 붙여 그립니다. 마지막 변수(ETT의 OT, 오일 온도) 기준입니다.

### 11-2. 분해 결과 (논문 Figure 4 형식)
마지막 디코더 층이 출력한 **계절 성분**과 **추세 성분**을 따로 그립니다. 두 성분을 더한 것이 최종 예측입니다.

### 11-3. 학습된 주기 (논문 Figure 6 형식)
테스트 구간의 각 샘플에 대해, 디코더의 inner Auto-Correlation이 상관이 크다고 판단한 **상위 10개 지연(lag)**을 모아 히스토그램으로 그립니다.
ETTm2는 15분 간격이므로 하루 주기는 96, 반나절은 48에 해당합니다.

In [ ]:
def load_trained(name, pred_len):
    cfg = build_config(name, pred_len)
    path = os.path.join(CKPT_DIR, f'{name}_{pred_len}.pth')
    if not os.path.exists(path):
        print('체크포인트가 없습니다. 먼저 섹션 9를 실행하세요:', path); return None, None, None
    model = Autoformer(cfg).float().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    sets, _ = make_loaders(cfg, num_workers=0)
    return model, cfg, sets['test']


@torch.no_grad()
def plot_prediction(name='ETTm2', pred_len=96, sample_ids=(0, 2000, 5000), var=-1):
    model, cfg, test_set = load_trained(name, pred_len)
    if model is None: return
    fig, axes = plt.subplots(1, len(sample_ids), figsize=(6 * len(sample_ids), 3.2))
    for ax, sid in zip(np.atleast_1d(axes), sample_ids):
        sid = min(sid, len(test_set) - 1)
        x, y, xm, ym = [t.unsqueeze(0).to(DEVICE) for t in test_set[sid]]
        pred = model(x, xm, ym)[0, :, var].cpu().numpy()
        hist = x[0, :, var].cpu().numpy(); true = y[0, -pred_len:, var].cpu().numpy()
        ax.plot(np.concatenate([hist, true]), label='GroundTruth', color='tab:blue')
        ax.plot(np.arange(len(hist), len(hist) + pred_len), pred, label='Prediction', color='tab:orange')
        ax.axvline(len(hist) - 1, color='gray', ls=':'); ax.set_title(f'{name} {cfg.seq_len}→{pred_len}, sample {sid}')
        ax.legend()
    plt.tight_layout(); save_fig(f'03_prediction_{name}_{pred_len}'); plt.show()


@torch.no_grad()
def plot_decomposition(name='ETTm2', pred_len=96, sample_id=0, var=-1):
    model, cfg, test_set = load_trained(name, pred_len)
    if model is None: return
    x, y, xm, ym = [t.unsqueeze(0).to(DEVICE) for t in test_set[sample_id]]
    out, seasonal, trend = model(x, xm, ym, return_parts=True)
    plt.figure(figsize=(10, 3.5))
    plt.plot(y[0, -pred_len:, var].cpu(), label='Ground truth', color='black')
    plt.plot(out[0, :, var].cpu(), label='Prediction (= seasonal + trend)', color='tab:orange')
    plt.plot(seasonal[0, :, var].cpu(), label='Seasonal part', color='tab:green')
    plt.plot(trend[0, :, var].cpu(), label='Trend-cyclical part', color='tab:blue')
    plt.title(f'Decomposition of the last decoder layer ({name} {cfg.seq_len}→{pred_len})'); plt.legend()
    save_fig(f'04_decomposition_{name}_{pred_len}'); plt.show()


@torch.no_grad()
def plot_learned_lags(name='ETTm2', pred_len=336, top=10, max_batches=None):
    model, cfg, test_set = load_trained(name, pred_len)
    if model is None: return
    ac = model.decoder.layers[0].self_attention.inner_correlation
    ac.record = True
    loader = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False)
    lags = []
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches: break
        x, y, xm, ym = to_device(batch)
        model(x, xm, ym)
        corr = ac.last_corr[:, 1:]                         # τ=0(자기 자신) 제외
        lags.append((torch.topk(corr, top, dim=-1).indices + 1).flatten())
    ac.record = False
    lags = torch.cat(lags).numpy()
    plt.figure(figsize=(10, 3.5))
    plt.hist(lags, bins=np.arange(0, cfg.label_len + pred_len + 2) - 0.5, density=True, color='tab:green')
    for p, lab in [(96, 'daily (96)'), (48, 'half-day (48)')]:
        plt.axvline(p, color='red', ls='--', lw=1); plt.text(p, plt.ylim()[1] * 0.9, lab, color='red')
    plt.xlabel('lag'); plt.ylabel('density')
    plt.title(f'Top-{top} lags learned by the decoder ({name} {cfg.seq_len}→{pred_len})')
    save_fig(f'05_learned_lags_{name}_{pred_len}'); plt.show()
    values, counts = np.unique(lags, return_counts=True)
    print('가장 자주 선택된 지연:', values[np.argsort(-counts)[:10]].tolist())


_vis_pred = DATASETS['ETTm2']['pred_lens'][0] if QUICK_TEST else 96
plot_prediction('ETTm2', _vis_pred)
if not QUICK_TEST:
    plot_prediction('ETTm2', 336)
plot_decomposition('ETTm2', _vis_pred)
plot_learned_lags('ETTm2', DATASETS['ETTm2']['pred_lens'][0] if QUICK_TEST else 336,
                  max_batches=5 if QUICK_TEST else None)

# 저장된 그래프를 zip 하나로 묶기 (Drive에서 한 번에 내려받기 쉽게)
import shutil
zip_path = shutil.make_archive(os.path.join(OUT_DIR, 'figures'), 'zip', FIG_DIR)
print('그래프 묶음:', zip_path)

---
### 논문과 공식 코드의 차이 정리 (발표용)
| # | 논문 | 공식 코드 (이 노트북이 따른 방식) |
|---|---|---|
| 1 | 식 2: 입력의 뒤쪽 절반만 분해 | 입력 전체를 분해한 뒤 마지막 `label_len`개를 사용 |
| 2 | 식 4: 추세마다 사영 $\mathcal{W}_{l,1}, \mathcal{W}_{l,2}, \mathcal{W}_{l,3}$ | 세 추세를 더한 뒤 사영 하나 (Conv1d) |
| 3 | Algorithm 4: 추론 시 배치 평균 지연 | 추론 시 샘플별 지연 |
| 4 | 학습률 스케줄 언급 없음 | epoch마다 절반 (`lradj=type1`) |
| 5 | 데이터별 epoch 언급 없음 | Weather·Traffic·Exchange 일부는 1~3 epoch만 학습 |
| 6 | ETT 6:2:2 분할 | 12개월 / 4개월 / 4개월 (한 달 = 30일) |